### Machine Learning | Assignment 3 - Evolutionary Algorithms

145873 | Sezer Tetik

#### Exercise 2

In [ ]:
import random

import numpy as np

**Setup from Exercise 1.** Each exercise is its own notebook, so the pattern generator from 1a is repeated here unchanged. The pattern size is the one chosen at the end of Exercise 1: **24 bits**, the largest size whose 30 brute-force trials finished within the time budget.

In [ ]:
SIZE = 24


def generate_random_pattern(size: int, rng: random.Random) -> str:
    if size < 1:
        raise ValueError("size must be >= 1")

    return format(rng.getrandbits(size), f"0{size}b")

**a)** Fitness of a guess: the number of positions where the guess and the correct pattern have the same bit.

$$f(g, t) = \sum_{i=1}^{n} \mathbb{1}[g_i = t_i] = n - d_H(g, t)$$

where $n$ is the pattern size and $d_H(g, t)$ is the Hamming distance, the number of positions where the two patterns differ.

- **Maximum only at the solution.** $f = n$ exactly when every bit matches, i.e. $g = t$.
- **Decreases with the difference.** Each wrong bit costs exactly 1 point, so fitness falls linearly with the Hamming distance, down to 0 for the complement of the target. Two guesses with the same number of wrong bits score the same, no matter which bits are wrong.
- **Mastermind feedback.** Like the black pegs in Mastermind, it says *how many* bits are right but not *which ones*. The search still has to work out which bits to change.

Brute force in Exercise 1 only used a yes/no check (`guess == target`). That check gives the same answer for all $2^n - 1$ wrong patterns, so a wrong guess carries no hint on where to go next. This fitness ranks every guess by how close it is to the target, which is what the search methods in the next questions rely on.

The fitness is the raw count (an integer from 0 to $n$) rather than the fraction $f/n$, so comparisons between guesses are exact, with no floating-point rounding. Dividing by $n$ gives the normalised version if different sizes ever need to be compared.

In [ ]:
def fitness(guess: str, target: str) -> int:
    if len(guess) != len(target):
        raise ValueError("guess and target must have the same size")

    return sum(g == t for g, t in zip(guess, target))

In [ ]:
rng = random.Random(0)
target = generate_random_pattern(SIZE, rng)
guess = generate_random_pattern(SIZE, rng)

print(f"target   {target}")
print(f"guess    {guess}")
print(f"matches  {''.join('|' if g == t else ' ' for g, t in zip(guess, target))}")
print(f"fitness  {fitness(guess, target)} / {SIZE}")

Sanity checks:

- the correct pattern gets the maximum $n$;
- flipping $k$ random bits of the target gives exactly $n - k$, for every $k$ from 0 to $n$. So fitness strictly decreases as the guess moves away from the target, and only the target reaches $n$;
- a random guess gets each bit right with probability 1/2, so its fitness follows a Binomial$(n, 1/2)$: mean $n/2 = 12$ and standard deviation $\sqrt{n}/2 \approx 2.45$. This is the typical starting fitness of a search that begins from a random pattern.

In [ ]:
rng = random.Random(1)
target = generate_random_pattern(SIZE, rng)

assert fitness(target, target) == SIZE

for k in range(SIZE + 1):
    flipped = set(rng.sample(range(SIZE), k))
    guess = "".join(("1" if bit == "0" else "0") if i in flipped else bit for i, bit in enumerate(target))
    assert fitness(guess, target) == SIZE - k

scores = np.array([fitness(generate_random_pattern(SIZE, rng), target) for _ in range(100_000)])

print(f"random guesses: mean {scores.mean():.2f} (theory {SIZE / 2:.2f}), std {scores.std():.2f} (theory {np.sqrt(SIZE) / 2:.2f})")